# Raw HMI Magnetogram Exploration

This notebook is a manual inspection aid for raw SDO/HMI FITS files. It is not part of the API runtime or training loop; use it when validating newly downloaded magnetograms before they enter `src/processing/prepare_dataset.py`.

The checks here focus on three things that commonly break ingestion runs: missing FITS files, unexpected metadata, and display ranges that hide active regions.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import sunpy.map

warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw')
fits_files = sorted(RAW_DIR.glob('*.fits'))

print(f'FITS files found: {len(fits_files)}')
for path in fits_files[:5]:
    print(f'  {path.name}')

## Load a Sample

`sunpy.map.Map` keeps the FITS image and WCS metadata together. That makes it the safest object to inspect before reducing the file to the 512 x 512 tensor format used by the model.

In [ ]:
if not fits_files:
    raise FileNotFoundError('No FITS files found under data/raw/. Run ingestion first.')

sample_path = fits_files[0]
solar_map = sunpy.map.Map(str(sample_path))

print(f'Loaded: {sample_path.name}')
print(f'Data shape: {solar_map.data.shape}')
print(f'Dtype: {solar_map.data.dtype}')

## Metadata Check

The training pipeline only needs the image array and a filename-derived timestamp, but metadata inspection is useful for catching corrupted downloads or unexpected instrument records.

In [ ]:
metadata_fields = {
    'instrument': solar_map.meta.get('INSTRUME', 'N/A'),
    'observation_date': solar_map.date,
    'spatial_scale_arcsec_px': solar_map.meta.get('CDELT1', 'N/A'),
    'wavelength': solar_map.meta.get('WAVELNTH', 'N/A'),
    'calibration_level': solar_map.meta.get('LVL_NUM', 'N/A'),
}

for key, value in metadata_fields.items():
    print(f'{key}: {value}')

## Field Statistics

Auralis uses a simple activity proxy based on the percentage of pixels with `|B| > 200 G`. The proxy should be computed before resizing, otherwise interpolation can soften compact active regions near the threshold.

In [ ]:
data = np.nan_to_num(solar_map.data.astype(np.float32), nan=0.0)
threshold = 200.0
active_positive = int(np.sum(data > threshold))
active_negative = int(np.sum(data < -threshold))
active_total = active_positive + active_negative

print('Magnetic field statistics')
print(f'  min:  {np.min(data):.2f} G')
print(f'  max:  {np.max(data):.2f} G')
print(f'  mean: {np.mean(data):.2f} G')
print(f'  std:  {np.std(data):.2f} G')
print()
print(f'Active pixels (|B| > {threshold:.0f} G): {active_total:,}')
print(f'Activity proxy: {100 * active_total / data.size:.4f}%')

## Magnetogram View

Use a fixed `+/-200 G` display range when inspecting active regions. Passing a `Normalize` object avoids the SunPy error that occurs when `vmin` or `vmax` conflicts with an existing map normalization.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection=solar_map)
norm = Normalize(vmin=-200, vmax=200)

solar_map.plot(
    axes=ax,
    cmap='hmimag',
    norm=norm,
    title=f'SDO/HMI Magnetogram - {solar_map.date.strftime("%Y-%m-%d %H:%M:%S")} UTC',
)
solar_map.draw_grid(axes=ax, color='white', alpha=0.4, linewidth=0.5)
cbar = plt.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.08)
cbar.set_label('Magnetic Field (Gauss)', rotation=270, labelpad=25)
plt.show()

## Distribution View

The histogram is mainly a sanity check. A healthy HMI magnetogram has most pixels near zero and long tails around active regions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(data.flatten(), bins=200, color='#3b82f6', alpha=0.75)
axes[0].set_yscale('log')
axes[0].axvline(-threshold, color='orange', linestyle='--', linewidth=1.5)
axes[0].axvline(threshold, color='orange', linestyle='--', linewidth=1.5)
axes[0].set_title('Full magnetic-field distribution')
axes[0].set_xlabel('Magnetic Field (Gauss)')
axes[0].set_ylabel('Pixel count')

zoom = data[(data >= -500) & (data <= 500)]
axes[1].hist(zoom, bins=100, color='#22c55e', alpha=0.75)
axes[1].axvline(-threshold, color='orange', linestyle='--', linewidth=1.5)
axes[1].axvline(threshold, color='orange', linestyle='--', linewidth=1.5)
axes[1].set_title('Zoomed distribution (+/-500 G)')
axes[1].set_xlabel('Magnetic Field (Gauss)')
axes[1].set_ylabel('Pixel count')

plt.tight_layout()
plt.show()